In [ ]:
from ib_insync import *
import pandas as pd
import numpy as np
import datetime
import xgboost as xgb
from sklearn.metrics import accuracy_score
import time

util.startLoop()  # solo en Jupyter / entornos con event loop activo

# Conectar a IBKR (asegúrate de tener TWS o IB Gateway corriendo)
ib = IB()
ib.connect('127.0.0.1', 4002, clientId=87)  # Ajusta el clientId si es necesario

In [30]:
# Define el contrato de la acción
symbol = 'GOOG'
contract = Stock(symbol, 'SMART', 'USD')

# Definir fechas para los datos
endDateTime = datetime.datetime.now().strftime("%Y%m%d %H:%M:%S")
durationStr = '10 M'  # últimos 5 días
barSizeSetting = '15 mins'
whatToShow = 'TRADES'
useRTH = True

# Target
TARGET_IN_BARS_AHEAD = 4

# Inference
PROB_THR = 0.7
CHECK_INTERVAL_MINUTES = 15 # Cada cuanto corre el bucle en minutos
STOP_LOSS_PCT = 0.98  # Stop-loss al 2% por debajo del precio de compra

def log(msg):
    """
    Imprime log con timestamp
    """
    print(f"[{pd.Timestamp.now()}] {msg}")

In [16]:
def getBars():

    # Descargar barras históricas
    bars = ib.reqHistoricalData(
        contract,
        endDateTime=endDateTime,
        durationStr=durationStr,
        barSizeSetting=barSizeSetting,
        whatToShow=whatToShow,
        useRTH=useRTH,
        formatDate=1
    )

    # Convertir bars a DataFrame
    data = []
    for bar in bars:
        dt = pd.Timestamp(bar.date)
        if dt.tzinfo is None:
            dt = dt.tz_localize('US/Eastern')  # Hora de NY (exchange)
        dt = dt.tz_convert('Europe/Madrid')    # Convertir a tu hora local si quieres
        data.append({
            'date': dt,
            'open': bar.open,
            'high': bar.high,
            'low': bar.low,
            'close': bar.close,
            'volume': bar.volume
        })

    df = pd.DataFrame(data)
    df.set_index('date', inplace=True)
    df.sort_index(inplace=True)
    
    return df

In [17]:
def createFeatures(data):
    
    df = data.copy()
    df.sort_index(ascending=True, inplace=True)
    
    # Feature: precios y volumen actuales
    df['return'] = df['close'].pct_change()
    df['volatility'] = df['close'].rolling(12).std()
    df['ma_12'] = df['close'].rolling(12).mean()
    df['ma_24'] = df['close'].rolling(24).mean()
    
    # Quitar filas con NaN
    df.dropna(inplace=True)

    return df

In [18]:
def createFeatures(data):

    df = data.copy()
    df.sort_index(inplace=True)

    # =========================
    # 1. RETURNS BASE
    # =========================
    df['ret_1'] = df['close'].pct_change()
    df['ret_3'] = df['close'].pct_change(3)
    df['ret_6'] = df['close'].pct_change(6)
    df['ret_12'] = df['close'].pct_change(12)

    # =========================
    # 2. VOLATILIDAD
    # =========================
    df['vol_12'] = df['ret_1'].rolling(12).std()
    df['vol_24'] = df['ret_1'].rolling(24).std()
    df['vol_48'] = df['ret_1'].rolling(48).std()

    # =========================
    # 3. MEDIAS / TENDENCIA LARGA
    # =========================
    for w in [12, 24, 36, 48, 72]:
        df[f'sma_{w}'] = df['close'].rolling(w).mean()
        df[f'dist_sma_{w}'] = (df['close'] - df[f'sma_{w}']) / df[f'sma_{w}']

    # slopes (pendiente tendencia)
    df['slope_24'] = df['sma_24'].diff()
    df['slope_48'] = df['sma_48'].diff()

    # régimen de tendencia
    df['trend_regime'] = (df['sma_24'] > df['sma_48']).astype(int)

    # =========================
    # 4. MOMENTUM / ROC
    # =========================
    df['roc_12'] = df['close'].pct_change(12)
    df['roc_24'] = df['close'].pct_change(24)

    # =========================
    # 5. RSI
    # =========================
    def rsi(series, period=14):
        delta = series.diff()
        up = delta.clip(lower=0)
        down = -delta.clip(upper=0)

        ma_up = up.rolling(period).mean()
        ma_down = down.rolling(period).mean()

        rs = ma_up / ma_down
        return 100 - (100 / (1 + rs))

    df['rsi_14'] = rsi(df['close'], 14)
    df['rsi_28'] = rsi(df['close'], 28)

    # =========================
    # 6. VOLUMEN
    # =========================
    df['vol_mean_24'] = df['volume'].rolling(24).mean()
    df['rel_volume'] = df['volume'] / df['vol_mean_24']

    df['vol_zscore'] = (
        (df['volume'] - df['volume'].rolling(24).mean()) /
        df['volume'].rolling(24).std()
    )

    # OBV (On Balance Volume)
    df['obv'] = (np.sign(df['ret_1']) * df['volume']).fillna(0).cumsum()

    # presión compradora simple
    df['buy_pressure'] = df['ret_1'] * df['volume']

    # =========================
    # 7. FEATURES TEMPORALES (muy útiles intradía)
    # =========================
    df['hour'] = df.index.hour
    df['minute'] = df.index.minute

    # =========================
    # 8. LIMPIEZA
    # =========================
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    return df

In [33]:
# Training
df_raw = getBars()
df = createFeatures(df_raw)
    
# Target: 1 si sube dentro de 12 barras, 0 si no
df['future_close'] = df['close'].shift(-TARGET_IN_BARS_AHEAD)
df['target'] = (df['future_close'] > df['close']).astype(int)

# dropna (no target)
df = df.dropna()

# Features y target
#feature_cols = ['open', 'high', 'low', 'close', 'volume', 'return', 'volatility', 'ma_12', 'ma_24']
X = df.drop(columns='target')
y = df['target']

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, shuffle=False  # Importante: no barajamos para series temporales
)

# Inicializar modelo
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    #use_label_encoder=True,
    eval_metric='logloss'
)

# Entrenar
model.fit(X_train, y_train)

# Evaluar train
y_pred = model.predict(X_train)
acc = accuracy_score(y_train, y_pred)
print(f'Accuracy train: {acc:.4f}')

# Evaluar test
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy test: {acc:.4f}')

Accuracy train: 0.9069
Accuracy test: 0.5610


In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_candles_basic(X):

    df = X.copy().sort_index()

    # Simular high/low
    high = df['high']
    low  = df['low']

    fig = go.Figure(
        go.Candlestick(
            x=df.index,
            open=df['open'],
            high=high,
            low=low,
            close=df['close'],
            name="Price"
        )
    )

    fig.update_layout(
        title="Velas 15m",
        xaxis_title="Time",
        yaxis_title="Price",
        xaxis_rangeslider_visible=False,
        height=600
    )

    fig.show()

plot_candles_basic(X_test)

In [15]:
import joblib

joblib.dump(model, 'xgb_model_ibkr.pkl')
print("Modelo guardado en xgb_model_ibkr.pkl")

Modelo guardado en xgb_model_ibkr.pkl


In [16]:
# Cargar modelo
model = joblib.load('xgb_model_ibkr.pkl')
print("Modelo cargado correctamente")

Modelo cargado correctamente


In [ ]:
def place_buy_order(qty=1):
    # Orden de compra
    order = MarketOrder('BUY', qty)
    trade = ib.placeOrder(contract, order)
    log(f"Orden de compra enviada: {trade}")

def setStopLoss(stop_price, qty=1):
        # Crear orden de stop-loss
        stop_order = StopOrder('SELL', qty, stopPrice=stop_price)
        ib.placeOrder(contract, stop_order)
        log(f"Stop-loss colocado a {stop_price}")

def has_open_position(ib, symbol='GOOG'):
    """
    Devuelve True si hay alguna posición abierta en el símbolo indicado, False si no.
    """
    contract = Stock(symbol, 'SMART', 'USD')
    ib.qualifyContracts(contract)
    
    positions = ib.positions()  # lista de todas las posiciones abiertas
    for pos in positions:
        if pos.contract.conId == contract.conId and pos.position != 0:
            return True
    return False

In [ ]:
while True:
    # Inference mode
    last_bars = getBars()
    X_latest = createFeatures(last_bars).iloc[-1:]
    price = X_latest['close'][0]

    # Predict
    proba = model.predict_proba(X_latest)[0][1]
    
    log("Latest bar:{} at price {} with probability {:.4f}".format(X_latest.index.format(), price, proba))

    # Check if open position
    if has_open_position(ib, symbol):
        log("Position already in place, no buy")
    elif proba > PROB_THR:
        log("Execute buy order")
        place_buy_order()
        setStopLoss(price * STOP_LOSS_PCT)
    else:
        log("No buy, low probability")

    time.sleep(CHECK_INTERVAL_MINUTES * 60)

[2026-01-31 17:55:50.203077] Latest bar:['2026-01-30 21:55:00+01:00'] at price 258.24 with probability 0.8011
[2026-01-31 17:55:50.352945] Position already in place, no buy
